# model-train-eval-toggle-around-sample — worked example 1: eval, no_grad, sample, restore train

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-train-eval-toggle-around-sample`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

To sample from a generator inside a training loop without corrupting BatchNorm running statistics, switch to `model.eval()`, run the forward inside `torch.no_grad()`, then restore `model.train()`. eval() makes BN use its frozen running stats and skip the update; no_grad() keeps the samples out of the autograd graph.

## Worked solution

We wrap a sampling forward in the standard BN-safe block.

1. **eval().** Puts BatchNorm into inference mode: it uses the stored `running_mean`/`running_var` and does NOT update them from this batch. Without this, sampling would shift the stats.
2. **no_grad().** Inside the context the forward builds no graph, so `samples.requires_grad` is False and no memory is wasted on backward bookkeeping.
3. **Forward.** `samples = model(noise)` runs cleanly under both switches.
4. **train().** Restore training mode so the next optimizer step behaves normally.

The demo snapshots BN's `running_mean` before and after sampling, prints that it is unchanged, that the model is back in train mode, and that samples carry no grad.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)

model = nn.Sequential(nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU(), nn.Linear(8, 4))
model.train()
# warm the BN running stats with a couple of train forwards
for _ in range(2):
    model(t.randn(16, 4)).sum().backward()

bn = model[1]
before = bn.running_mean.clone()

def sample_clean(model, noise):
    model.eval()
    with t.no_grad():
        samples = model(noise)
    model.train()
    return samples

samples = sample_clean(model, t.randn(16, 4))
print('samples requires_grad:', samples.requires_grad)
print('back in train mode:', model.training)
print('BN running_mean unchanged:', t.allclose(before, bn.running_mean))